# Milestone 3 lab, notebook 8 of 8: The production version

## Where we are

| Notebook | What you do |
|---|---|
| 1. One order, one row | Write the same rows three ways: append, overwrite, upsert |
| 2. Which column is the key? | Merge on the wrong key. Then meet an order that arrives twice in one run |
| 3. Run it twice | Prove that a second run changes nothing |
| 4. At least once, exactly once | Crash the load in three places, and see what each one costs |
| 5. The update that came late | Find a change the watermark has passed. Catch it with a lookback |
| 6. Fix one day | Break Friday's numbers, and rebuild Friday alone |
| 7. The order that vanished | A deleted order, and why the query cannot see it |
| **8. The production version** (you are here) | The three properties, as tests |

You have used the finished pipeline since notebook 3. This notebook walks through it, and through the tests that prove it is safe.

In [1]:
import inspect

from lab_helpers import lab_warehouse, run_pipeline, show_doc, table

from daily_orders import load, pipeline
from daily_orders.report import format_answer

## Step 1: The parts

In [2]:
show_doc("LLD.md", "1. Files")

**From `docs/LLD.md`**

> ## 1. Files
> 
> Everything is inside `src/m3_safe_to_run_twice/`. Nothing imports from another milestone.
> 
> ```
> docs/                     BRD.md, HLD.md, LLD.md
> notebooks/                the lab: eight notebooks, run in order
> daily_orders/             production code
>   config.py               addresses, the retry settings, LOOKBACK_DAYS
>   extract.py              changed orders (with retry), and deletes from the change feed
>   warehouse.py            the Postgres database and its tables
>   load.py                 land raw, stage, MERGE, soft delete, explain, save progress
>   report.py               rebuild the report day by day; read it
>   pipeline.py             one morning, in one transaction
>   flow_data.py            builds the numbers shown on the flow page and the recap page
> tests/                    the three properties, and everything else
> milestone_3_flow.html     the whole pipeline, animated
> milestone_3_recap.html    the closing page
> ```

The heart is one statement. You wrote a version of it by hand in notebook 1.

In [3]:
print(load.MERGE_SQL)


MERGE INTO core.orders AS t
USING stage_orders AS s
   ON t.order_id = s.order_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN          -- a newer version: update the row
    UPDATE SET status         = s.status,
               item_count     = s.item_count,
               total_cents    = s.total_cents,
               refunded_cents = s.refunded_cents,
               updated_at     = s.updated_at,
               last_run_id    = %(run_id)s
WHEN NOT MATCHED THEN                                       -- a new order: insert a row
    INSERT (order_id, customer_id, city, status, placed_at, order_date, item_count,
            total_cents, refunded_cents, updated_at, first_run_id, last_run_id)
    VALUES (s.order_id, s.customer_id, s.city, s.status, s.placed_at, s.order_date, s.item_count,
            s.total_cents, s.refunded_cents, s.updated_at, %(run_id)s, %(run_id)s)



And here is one whole morning. Everything after the extract is inside `with con.transaction()`: it is all saved, or none of it.

In [4]:
print(inspect.getsource(pipeline.run))

def run(
    client: httpx.Client,
    con: psycopg.Connection,
    sleep: Callable[[float], None] = time.sleep,
    lookback_days: int = LOOKBACK_DAYS,
) -> RunResult:
    """Run the pipeline once, for the morning on the order system's clock."""
    run_date = read_run_date(client)
    watermark = read_watermark(con)
    # Lookback: start a little before the watermark, to catch updates that arrived late.
    # It re-reads rows we already hold. That is safe only because the write is idempotent.
    since = watermark - timedelta(days=lookback_days) if watermark else None

    # 1. Read from the source. If this fails, we stop here and nothing below happens.
    extract = fetch_changed_orders(client, to_api_text(since), sleep)
    deletes, feed_position = fetch_deletes(client, read_feed_position(con))

    # 2. Write. Everything is in ONE transaction: all of it is saved, or none of it.
    previous = con.execute("SELECT max(run_date) FROM ops.load_runs").fetchone()[0]
    with con.transac

## Step 2: Run it

In [5]:
con = lab_warehouse(fresh=True)
result = run_pipeline(con, "2026-06-08", "2026-06-09")

print(f"Run {result.run_id}: asked for orders changed since {result.asked_since}.")
print(f"Landed {result.rows_landed} rows. Orders: {result.orders_inserted} new, {result.orders_updated} updated. "
      f"Rebuilt {len(result.days_rebuilt)} days of the report.\n")
print(format_answer(result.answer))

Run 2026-06-09_1: asked for orders changed since 2026-06-06T03:48:53Z.
Landed 102 rows. Orders: 25 new, 4 updated. Rebuilt 4 days of the report.

Arden daily orders. Report for the morning of Tue 2026-06-09

  Yesterday   Mon 2026-06-08    25 orders      $591.75
  Day before  Sun 2026-06-07    30 orders      $807.11
  Change                         -5 orders     -$215.36

  Earlier days that changed in this run:
    Fri 2026-06-05   +0 orders   -$5.78
      because order 1130: refund of $5.78


The fourth part of the manager's question is answered again, and this time from a log the pipeline keeps.

In [6]:
table(con, "SELECT * FROM ops.report_changes ORDER BY run_id")

,run_id,order_date,order_id,what,orders_change,revenue_cents_change
0,2026-06-09_1,2026-06-05,1130,refund of $5.78,0,-578


In [7]:
table(con, "SELECT run_id, asked_since, rows_landed, orders_inserted, orders_updated, orders_deleted, api_calls FROM ops.load_runs ORDER BY run_id")

,run_id,asked_since,rows_landed,orders_inserted,orders_updated,orders_deleted,api_calls
0,2026-06-08_1,NaT,226,226,0,0,10
1,2026-06-09_1,2026-06-06 03:48:53+00:00,102,25,4,0,5


In [8]:
con.close()

## Step 3: The three properties, as tests

The business asked for three things, each proved by a test and not by a claim.

In [9]:
show_doc("BRD.md", "6. Success criteria")

**From `docs/BRD.md`**

> ## 6. Success criteria
> 
> Three properties, each proved by a test and not by a claim.
> 
> | # | Property | The proof |
> |---|---|---|
> | 1 | **Run it twice and nothing changes** | Run 9 June twice, then five times. The orders table and the report are identical to one run |
> | 2 | **Re-run an old day and that day is corrected** | Break Friday's numbers by hand. Rebuild Friday. It is right again, and no other day was touched |
> | 3 | **An order cancelled later lowers the old day's total** | Order 1091, placed 3 June, is cancelled on 4 June. The next run lowers 3 June by 1 order and $20.56 |
> 
> And the report must equal the answer key **on every morning from 2 June to 18 June**. In milestone 2 it was wrong from the second morning on.

In [10]:
!cd .. && uv run pytest --color=no -v -k "twice or five_runs or rebuilding or cancelled_later or every_morning" 2>&1 | grep -E "PASSED|FAILED" | sed -e 's/tests\/test_pipeline.py:://' -e 's/ *\[.*//'

tests/test_load.py::test_an_order_that_arrives_twice_in_one_run_is_merged_once PASSED
test_running_the_same_morning_twice_changes_nothing PASSED
test_five_runs_in_a_row_equal_one PASSED
test_rebuilding_one_day_corrects_it_and_touches_no_other PASSED
test_an_order_cancelled_later_lowers_the_old_days_total PASSED
test_the_report_equals_the_answer_key_on_every_morning PASSED


And all the tests of this milestone:

In [11]:
!cd .. && uv run pytest -q --color=no 2>&1 | tail -n 2

.............................                                            [100%]
29 passed in 4.02s


In [12]:
show_doc("LLD.md", "8. Checks")

**From `docs/LLD.md`**

> ## 8. Checks
> 
> | Property | The test |
> |---|---|
> | Run it twice and nothing changes | `test_running_the_same_morning_twice_changes_nothing`, `test_five_runs_in_a_row_equal_one` |
> | Re-run an old day and that day is corrected | `test_rebuilding_one_day_corrects_it_and_touches_no_other` |
> | An order cancelled later lowers the old day's total | `test_an_order_cancelled_later_lowers_the_old_days_total` |
> | The report equals the answer key on every morning, 2 to 18 June | `test_the_report_equals_the_answer_key_on_every_morning` |
> | A repeated page changes nothing | `test_fixed_a_repeated_page_changes_nothing` |
> | The late update is caught, and missed without a lookback | `test_fixed_the_late_update_is_caught_by_the_lookback`, `test_without_a_lookback_...` |
> | A deleted order is marked, kept and not counted | `test_fixed_a_deleted_order_is_marked_and_not_counted` |
> | A crash in the middle saves nothing | `test_a_crash_in_the_middle_saves_nothing_and_the_next_run_recovers` |
> | Without the dedupe step, `MERGE` refuses a repeated page | `test_without_the_dedupe_step_merge_refuses_the_repeated_page` |
> 
> All are in `tests/test_pipeline.py` and `tests/test_load.py`.

In milestone 2, four tests pinned down what was **wrong**. Here each of them has turned around: `test_fixed_a_repeated_page_changes_nothing`, `test_fixed_the_late_update_is_caught_by_the_lookback`, and so on.

## Step 4: What is still wrong

The numbers are right on every morning. So what is left?

In [13]:
show_doc("LLD.md", "9. What is still wrong")

**From `docs/LLD.md`**

> ## 9. What is still wrong
> 
> The numbers are right. The way they are made is not.
> 
> | Problem | What it looks like here |
> |---|---|
> | All the logic is in one script | The report's rules are SQL inside Python strings in `load.py` and `report.py`. A person who knows SQL but not Python cannot change them |
> | Nothing checks the tables | The tests prove the **code** on test data. Nothing looks at the tables after this morning's real run |
> | Nobody knows when the number is wrong | If the order system sent a negative total tomorrow, the report would add it up and say nothing |
> 
> **All the logic is in one script. Nothing checks the tables. Nobody knows when the number is wrong.** Milestone 4 moves the SQL into models and puts tests on the tables.

Look at `load.py` once more. The rule "a cancelled order is not a sale" is a line of SQL inside a Python string. The rule for the business day is another. To change one, you edit program code.

And every test here runs on **test** data. Tomorrow morning the real order system could send a status nobody has seen, or a negative total. The pipeline would load it, add it up, and report it.

## The words of milestone 3

| Word | In one or two sentences | You saw it in |
|---|---|---|
| **Upsert**, **MERGE** | Update the row if the key exists, add it if not. `MERGE` is the SQL statement for it | Notebook 1 |
| **Overwrite** | Empty the table and load everything. Right for a full load only | Notebook 1 |
| **Primary key**, **natural key** | The column the database keeps unique. A key that comes from the business itself | Notebook 2 |
| **Dedupe key** | The key you keep one row per, when a thing arrived more than once | Notebook 2 |
| **Idempotent**, **safe re-run** | Two runs leave the same result as one run | Notebook 3 |
| **At-most-once**, **at-least-once**, **exactly-once** | A row can be lost. A row can repeat. Each row changes the result one time | Notebook 4 |
| **Late row**, **lookback** | A row that arrives after the window that should have caught it. Re-reading N days before the watermark | Notebook 5 |
| **Partition**, **reprocessing** | One slice of a table, here one day. Building a past slice again | Notebook 6 |
| **Hard delete**, **soft delete** | Remove the row for good. Keep the row and mark it | Notebook 7 |
| **Log-based CDC** | Capturing changes from the database's own log. It sees deletes | Notebook 7 |

The project's [`GLOSSARY.md`](../../../GLOSSARY.md) keeps every word, with an interview line for each.

## What you now know

- One order is one row, however often it arrives. A second run changes nothing.
- One transaction holds the whole write, so a crash leaves the warehouse as it was.
- Each property is a test you can run, not a sentence you must believe.

**You have finished milestone 3.** Open [`milestone_3_recap.html`](../milestone_3_recap.html) in a browser. Then open [`milestone_3_flow.html`](../milestone_3_flow.html) and press **Run it again**.

> **Still wrong:** all the logic is in one script. Nothing checks the tables. Nobody knows when the number is wrong. Milestone 4 moves the SQL into models, and puts tests on the tables.